In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import re

# Load the dataset
try:
    df = pd.read_csv('/content/processed_data (1).csv')
    print("Dataset loaded successfully.")
    print(f"Dataset shape: {df.shape}")
except FileNotFoundError:
    print("Error: 'processed_data (1).csv' not found. Please ensure the file is uploaded or accessible.")
    df = None

if df is not None:
    # Split the dataset into train and test sets
    # Assuming the target variable is not specified, so splitting the entire dataframe.
    # If there's a target column, it should be adjusted accordingly.
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=0)

    print(f"\nTrain set shape: {train_df.shape}")
    print(f"Test set shape: {test_df.shape}")

    # Determine the number of unique words in the text column of the train set
    if 'text' in train_df.columns:
        all_words = []
        for text in train_df['text'].astype(str):
            # Convert to lowercase and split into words using regex to handle punctuation
            words = re.findall(r'\b\w+\b', text.lower())
            all_words.extend(words)

        unique_words = set(all_words)
        num_unique_words = len(unique_words)

        print(f"\nNumber of unique words in the 'text' column of the train set: {num_unique_words}")
    else:
        print("Error: 'text' column not found in the training DataFrame.")


Dataset loaded successfully.
Dataset shape: (16363, 8)

Train set shape: (13090, 8)
Test set shape: (3273, 8)

Number of unique words in the 'text' column of the train set: 16446


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer

# Define feature groups based on the prompt
numerical_features = ['Land Area (Km²)']
ordinal_features = ['Density_Level', 'Population_Group']
nominal_features = ['Age of User', 'Time of Tweet', 'Continent']
text_feature = 'text'

# --- Preprocessing Steps ---

# 1. Numerical Feature (Land Area (Km²))
numeric_transformer = StandardScaler()

# 2. Ordinal Features (Density_Level, Population_Group)
# The categories must be specified in the order of mapping: "Low" -> 0, "Medium" -> 1, "High" -> 2
ordinal_transformer = OrdinalEncoder(categories=[['Low', 'Medium', 'High'], ['Low', 'Medium', 'High']])

# 3. Nominal Features (Age of User, Time of Tweet, Continent)
nominal_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')

# 4. Text Feature (text)
text_transformer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b\w\w+\b|[@#]\w+', # Includes hashtags and mentions
    strip_accents='unicode' # Normalizes characters like "é"
)

# Create a ColumnTransformer to apply different transformers to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('ord', ordinal_transformer, ordinal_features),
        ('nom', nominal_transformer, nominal_features),
        ('text_vec', text_transformer, text_feature) # TfidfVectorizer expects a single column name
    ],
    remainder='drop' # Drop any columns not specified in transformers
)

# Fit the preprocessor on the training data and transform both training and test data
X_train_transformed = preprocessor.fit_transform(train_df)
X_test_transformed = preprocessor.transform(test_df)

print(f"Shape of transformed training data: {X_train_transformed.shape}")
print(f"Shape of transformed test data: {X_test_transformed.shape}")

# Calculate the sum of all values in the first five rows of the transformed test feature matrix
sum_first_five_rows_test = np.sum(X_test_transformed[:5, :])

print(f"\nSum of all values in the first five rows of transformed test feature matrix: {sum_first_five_rows_test:.2f}")


Shape of transformed training data: (13090, 5016)
Shape of transformed test data: (3273, 5016)

Sum of all values in the first five rows of transformed test feature matrix: 26.89


In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import log_loss

# Extract the target variable 'sentiment'
y_train = train_df['sentiment']
y_test = test_df['sentiment']

# Exclude the 'Land Area (Km²)' column from the transformed features
# This column corresponds to the first column (index 0) in X_train_transformed and X_test_transformed.
X_train_nb = X_train_transformed[:, 1:]
X_test_nb = X_test_transformed[:, 1:]

print(f"Shape of X_train for MultinomialNB (excluding Land Area): {X_train_nb.shape}")
print(f"Shape of X_test for MultinomialNB (excluding Land Area): {X_test_nb.shape}")

# Initialize and train the MultinomialNB model
model = MultinomialNB()
model.fit(X_train_nb, y_train)

print("\nMultinomialNB model trained successfully.")

# Predict probabilities on the test set
y_pred_proba = model.predict_proba(X_test_nb)

# Calculate log_loss
# MultinomialNB outputs probabilities for each class, so we need to ensure y_test is correctly mapped to numerical labels
# Assuming 'negative' is class 0 and 'positive' is class 1 (or vice versa depending on internal encoding)
# log_loss expects probabilities for all classes. The classes in y_train/y_test are 'negative' and 'positive'.
# Let's verify the order of classes in the model.

# Get the class labels from the model to ensure correct order for log_loss
class_labels = model.classes_

# Map y_test string labels to integer labels based on model.classes_
y_test_encoded = y_test.apply(lambda x: list(class_labels).index(x))

ll = log_loss(y_test_encoded, y_pred_proba)

print(f"\nLog_loss on the test dataset: {ll:.2f}")


Shape of X_train for MultinomialNB (excluding Land Area): (13090, 5015)
Shape of X_test for MultinomialNB (excluding Land Area): (3273, 5015)

MultinomialNB model trained successfully.

Log_loss on the test dataset: 0.37


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Initialize and train the RandomForestClassifier with random_state=42
# Use X_train_transformed and y_train (which are already defined from previous steps)
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_transformed, y_train)

print("RandomForestClassifier trained successfully (including Land Area (Km²)).\n")

# Make predictions on the preprocessed test dataset
y_pred_rf = rf_model.predict(X_test_transformed)

# Evaluate the model and find the confusion matrix
cm = confusion_matrix(y_test, y_pred_rf, labels=rf_model.classes_)

print("Confusion Matrix:")
print(f"True Negative (negative as negative): {cm[0, 0]}")
print(f"False Positive (negative as positive): {cm[0, 1]}")
print(f"False Negative (positive as negative): {cm[1, 0]}")
print(f"True Positive (positive as positive): {cm[1, 1]}")

# Interpret confusion matrix to find the most confusing class
# Assuming classes are ordered as 'negative', 'positive' as per model.classes_
# Misclassifications for 'negative' class: True 'negative' predicted as 'positive' (cm[0, 1])
# Misclassifications for 'positive' class: True 'positive' predicted as 'negative' (cm[1, 0])

negative_misclassified_as_positive = cm[0, 1]
positive_misclassified_as_negative = cm[1, 0]

if negative_misclassified_as_positive > positive_misclassified_as_negative:
    most_confusing_class = 'negative' # More actual negatives were mistaken for positives
    confusion_count = negative_misclassified_as_positive
elif positive_misclassified_as_negative > negative_misclassified_as_positive:
    most_confusing_class = 'positive' # More actual positives were mistaken for negatives
    confusion_count = positive_misclassified_as_negative
else:
    most_confusing_class = 'Both classes were equally confusing' # Equal number of misclassifications
    confusion_count = negative_misclassified_as_positive

print(f"\nThe model found the '{most_confusing_class}' class most confusing, with {confusion_count} misclassifications.")

# Optional: print full classification report for more details
# print("\nClassification Report:")
# print(classification_report(y_test, y_pred_rf, target_names=rf_model.classes_))


RandomForestClassifier trained successfully (including Land Area (Km²)).

Confusion Matrix:
True Negative (negative as negative): 1348
False Positive (negative as positive): 184
False Negative (positive as negative): 267
True Positive (positive as positive): 1474

The model found the 'positive' class most confusing, with 267 misclassifications.


In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

# Initialize the Logistic Regression estimator
logistic_estimator = LogisticRegression(random_state=42, max_iter=1000)

# Initialize RFECV with the specified parameters
# We'll use 'accuracy' as the scoring metric for classification problems
rfecv = RFECV(estimator=logistic_estimator, step=100, cv=3, scoring='accuracy', n_jobs=-1)

# Fit RFECV on the preprocessed training data
rfecv.fit(X_train_transformed, y_train)

# Get the number of selected features
num_selected_features = rfecv.n_features_

print(f"Number of features selected by RFECV: {num_selected_features}")


Number of features selected by RFECV: 2116
